<a id="top"></a>

# Demo: build the shallow agent in one line with `create_agent`

```yaml
title: "Demo: build the shallow agent in one line with create_agent"
keywords: create_agent, langchain, langgraph, ChatAnthropic, FakeModel, shallow agent, chaining task, flaky_fetch, behavioral equivalence, messages, tool_calls, ToolMessage, recursion limit, draw_mermaid, sonnet, teacher demo
estimated duration: 25
```

> **Lesson:** L11. Teacher demo — the runnable core of the lesson, paired with the [L1102 slide outline](L1102_lecture.md) (the graph you're about to build) and the [L1105 slide outline](L1105_lecture.md) (its config surface).
>
> **Runs two ways with the same code.** By default it drives the course's scripted **`FakeModel`** (offline, deterministic, no key) so a restart-and-run-all always passes. Set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) and the **live** cells drive a real **`ChatAnthropic`** (Claude **Sonnet 4.6**); with no key they print a skip line and the notebook still completes.
>
> The point to land: **`create_agent` is your L10 loop, packaged.** One call returns a running shallow agent — model → tool → model until termination — with the loop, routing, message bookkeeping, and step cap all supplied for you.

## Contents

- [1. Setup — the shared tools and the two models](#1-setup--the-shared-tools-and-the-two-models)
- [2. Build the agent in one line](#2-build-the-agent-in-one-line)
- [3. Run 1 — the chaining task (behavioral equivalence)](#3-run-1--the-chaining-task-behavioral-equivalence)
- [4. Read the returned messages](#4-read-the-returned-messages)
- [5. Run 2 — the flaky_fetch failure task](#5-run-2--the-flaky_fetch-failure-task)
- [6. Render what it wrapped — the graph](#6-render-what-it-wrapped--the-graph)
- [7. Name the freebies against their L10 twins](#7-name-the-freebies-against-their-l10-twins)
- [8. (Optional) the same agent on a live model](#8-optional-the-same-agent-on-a-live-model)
- [9. Takeaways](#9-takeaways)

## 1. Setup — the shared tools and the two models

We import the **same** `calculator`, `lookup`, and `flaky_fetch` tools students already met in L10 (from `common/tools.py`) — rebuilding a *known* agent is the whole point, so the only new thing on screen is the one-liner. `create_agent` takes a plain list of callables and infers each tool's schema from its type hints + docstring, exactly like `bind_tools` did in L10.

For the model we have two interchangeable options:
- **`FakeModel`** (from `common/fake_model.py`): a *scripted* stand-in that returns pre-decided replies. It makes this notebook deterministic and keyless — the default path.
- **`ChatAnthropic`** (Sonnet 4.6): the real client, used in the optional live section. Note this is the **native LangChain client**, not the repo's `potato_llm` seam — *frameworks bring their own client abstraction.* The key still loads through `common/config.py`.

In [1]:
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

from fluffy_potato_curriculum.common.config import get_settings
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)
from fluffy_potato_curriculum.common.tools import calculator, flaky_fetch, lookup

# The course anchor model. ChatAnthropic wants the bare id (no 'anthropic:' prefix).
SONNET = "claude-sonnet-4-6"
# Live cells are gated on a present key, so a keyless run still completes.
LIVE = bool(get_settings().anthropic_api_key)

# The same three tools from L10, imported (not rebuilt).
TOOLS = [calculator, lookup, flaky_fetch]

SYSTEM_PROMPT = (
    "You are a precise assistant. Use the provided tools to answer; "
    "do not answer from memory. When you have the answer, state it plainly."
)

print("tools:", [t.__name__ for t in TOOLS])
print("live model:", SONNET if LIVE else "(no ANTHROPIC_API_KEY — the live section will skip)")

tools: ['calculator', 'lookup', 'flaky_fetch']
live model: (no ANTHROPIC_API_KEY — the live section will skip)


[↑ Back to top](#top)

## 2. Build the agent in one line

Here is the entire lesson. One call to `create_agent` — a model, a list of tools, a system prompt — returns a **running shallow agent**. Say out loud what you did **not** write: no `while` loop, no `if reply.tool_calls` branch, no `messages.append(...)` bookkeeping, no step counter, no tool dispatch. All of it is inside the object `create_agent` just handed back.

We build it on the `FakeModel` first so the run below is deterministic. The `FakeModel` is scripted to issue the *same* tool sequence the L10 loop produced on this task — `calculator` then `lookup` — then answer in plain text.

In [2]:
# Scripted replies: calculator(17**2 - 1) -> 288, lookup('Lagos') -> 15000000, then a text
# answer. This is the SAME sequence the L10 hand-rolled loop produced on the chaining task.
chaining_script = [
    tool_reply(tool_call("c1", "calculator", {"expression": "17**2 - 1"})),
    tool_reply(tool_call("c2", "lookup", {"city": "Lagos"})),
    text_reply("The city is Lagos (17**2 - 1 = 288), and its population is 15,000,000."),
]
fake_model = FakeModel(chaining_script)

# THE WHOLE LESSON: one line. model + tools + system prompt -> a running shallow agent.
agent = create_agent(fake_model, TOOLS, system_prompt=SYSTEM_PROMPT)

print("built:", type(agent).__name__)  # a compiled LangGraph graph — the loop, packaged

built: CompiledStateGraph


[↑ Back to top](#top)

## 3. Run 1 — the chaining task (behavioral equivalence)

The **L10 chaining task**: *the population of the city whose name is the answer to `17**2 - 1`.* This forces a chain — compute with `calculator`, feed the result into `lookup`, then answer — and terminates *naturally* when the model stops asking for tools.

You invoke a `create_agent` agent by passing a `messages` list. It returns a dict whose `messages` key is the full conversation, tool calls and all — the same message history L10 built by hand.

In [3]:
chaining_task = (
    "What is the population of the city whose name is the answer to 17**2 - 1? "
    "Use the calculator first, then look the city up."
)

result = agent.invoke({"messages": [HumanMessage(content=chaining_task)]})

# Watch the tool sequence: which tools ran, in what order.
tool_sequence = [
    call["name"]
    for msg in result["messages"]
    if isinstance(msg, AIMessage)
    for call in msg.tool_calls
]
print("tool sequence:", tool_sequence)  # expect ['calculator', 'lookup'] — same as L10

tool sequence: ['calculator', 'lookup']


[↑ Back to top](#top)

## 4. Read the returned messages

The returned `messages` list is the agent's whole run. It is the *same* user / assistant-with-tool-calls / tool-result / assistant sequence the L10 loop appended by hand — you just didn't write the appends. Print it turn by turn, then pull the final answer off the last message.

In [4]:
def describe(msg: object) -> str:
    """One readable line per message: its type, its text, and any tool calls."""
    kind = type(msg).__name__
    if isinstance(msg, AIMessage) and msg.tool_calls:
        calls = ", ".join(f"{c['name']}({c['args']})" for c in msg.tool_calls)
        return f"{kind:13} -> tool call: {calls}"
    if isinstance(msg, ToolMessage):
        return f"{kind:13} <- result [{msg.status}]: {msg.content!r}"
    text = msg.content if isinstance(msg.content, str) else str(msg.content)
    return f"{kind:13}    {text!r}"


for msg in result["messages"]:
    print(describe(msg))

# The final answer is the content of the last message (a plain-text AIMessage).
final_answer = result["messages"][-1].content
print("\nfinal answer:", final_answer)

HumanMessage     'What is the population of the city whose name is the answer to 17**2 - 1? Use the calculator first, then look the city up.'
AIMessage     -> tool call: calculator({'expression': '17**2 - 1'})
ToolMessage   <- result [success]: '288'
AIMessage     -> tool call: lookup({'city': 'Lagos'})
ToolMessage   <- result [success]: '15000000'
AIMessage        'The city is Lagos (17**2 - 1 = 288), and its population is 15,000,000.'

final answer: The city is Lagos (17**2 - 1 = 288), and its population is 15,000,000.


[↑ Back to top](#top)

## 5. Run 2 — the flaky_fetch failure task

The **L10 `flaky_fetch` failure task**. `flaky_fetch` fails in two distinct ways: some URLs **return an error as data** (the call succeeds; the *content* says it failed — the L08 pattern), and one URL **raises**. Here we script the model to hit `https://error` (which returns a `503` error string), notice the failure in the tool result, and **retry** `https://ok` — exactly the recover-from-a-bad-tool-result behavior L10 taught.

> **One real boundary difference, worth naming.** By *default* `create_agent`'s tool node **re-raises** an uncaught Python exception — unlike your L10 loop, which caught every exception and turned it into an error `ToolMessage`. So we drive the *error-as-data* path here (the tool returns, the content is bad), which both the L10 loop and `create_agent` handle identically. The crashing URL (`https://crash`) would need error-handling configured on the tool node — a detail that lives below the one-liner (L15).

In [5]:
# https://error returns '{"error": "503 ..."}' (a failure as data, not an exception);
# the model reads that, then retries https://ok, which returns a usable body.
flaky_script = [
    tool_reply(tool_call("f1", "flaky_fetch", {"url": "https://error"})),
    tool_reply(tool_call("f2", "flaky_fetch", {"url": "https://ok"})),
    text_reply("https://error returned a 503, so I retried https://ok: the answer is 42."),
]
flaky_agent = create_agent(FakeModel(flaky_script), TOOLS, system_prompt=SYSTEM_PROMPT)

flaky_task = (
    "Fetch https://error. If the result looks like a failure, retry https://ok, "
    "then tell me what you got."
)
flaky_result = flaky_agent.invoke({"messages": [HumanMessage(content=flaky_task)]})

for msg in flaky_result["messages"]:
    print(describe(msg))

HumanMessage     'Fetch https://error. If the result looks like a failure, retry https://ok, then tell me what you got.'
AIMessage     -> tool call: flaky_fetch({'url': 'https://error'})
ToolMessage   <- result [success]: '{"error": "503 service unavailable"}'
AIMessage     -> tool call: flaky_fetch({'url': 'https://ok'})
ToolMessage   <- result [success]: 'the answer is 42'
AIMessage        'https://error returned a 503, so I retried https://ok: the answer is 42.'


[↑ Back to top](#top)

## 6. Render what it wrapped — the graph

`create_agent` returns a compiled graph. Ask it to draw itself and you get *exactly* the diagram from the [L1102 slide outline](L1102_lecture.md): an `agent` node, a `tools` node, a **back-edge** `tools → agent`, and a conditional exit out of `agent` (tool call → `tools`; plain text → `__end__`). The one-liner **is** the graph you drew.

We print the Mermaid text (renders offline, no network). In a live class you can call `agent.get_graph().draw_mermaid_png()` to render an image if the draw path is available.

> **Node names.** LangGraph labels the model-call node **`model`** and the tool node **`tools`** in the rendered graph. The [L1102 outline](L1102_lecture.md) calls the model-call node the `agent` node — same node, and `agent`/`model` are used interchangeably for it throughout the lesson.

In [6]:
graph = agent.get_graph()
print(graph.draw_mermaid())

# The nodes and the all-important back-edge, read straight off the compiled graph:
print("\nnodes:", list(graph.nodes))
edges = [(e.source, e.target) for e in graph.edges]
print("tools -> agent back-edge present:", ("tools", "model") in edges)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools -.-> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


nodes: ['__start__', 'model', 'tools', '__end__']
tools -> agent back-edge present: True


[↑ Back to top](#top)

## 7. Name the freebies against their L10 twins

Everything the run above did that you did **not** write — each is a line you *did* write by hand in L10. That trade — you give up the knobs, you get the boilerplate for free — is the whole value proposition.

In [7]:
freebies = [
    ("run-driver loop", "your `while step in range(max_steps)`"),
    ("tool-call routing", "your `if reply.tool_calls:` branch"),
    ("message-history append", "your manual `messages.append(reply / ToolMessage)`"),
    ("tool execution", "your `dispatch(tools, call)`"),
    ("recursion / step limit", "your `max_steps` cap (LangGraph default: 25 steps)"),
]
print(f"{'create_agent gives you':26}  {'L10 twin you wrote by hand'}")
print("-" * 78)
for freebie, twin in freebies:
    print(f"{freebie:26}  {twin}")

create_agent gives you      L10 twin you wrote by hand
------------------------------------------------------------------------------
run-driver loop             your `while step in range(max_steps)`
tool-call routing           your `if reply.tool_calls:` branch
message-history append      your manual `messages.append(reply / ToolMessage)`
tool execution              your `dispatch(tools, call)`
recursion / step limit      your `max_steps` cap (LangGraph default: 25 steps)


[↑ Back to top](#top)

## 8. (Optional) the same agent on a live model

Swap the `FakeModel` for a real `ChatAnthropic` (Sonnet 4.6) and the **same** `create_agent` call drives a live model — same interchangeability you relied on in L10. A real model *varies*, so the exact tool order can differ run to run; the point is that the *code* is identical. This cell is gated on a key: with none set it prints a skip line so a keyless run still passes.

**Clear this cell's output before committing** — live output is non-deterministic.

In [8]:
if LIVE:
    from langchain_anthropic import ChatAnthropic

    live_model = ChatAnthropic(model=SONNET)  # reads ANTHROPIC_API_KEY from the environment
    live_agent = create_agent(live_model, TOOLS, system_prompt=SYSTEM_PROMPT)
    live_result = live_agent.invoke({"messages": [HumanMessage(content=chaining_task)]})
    for msg in live_result["messages"]:
        print(describe(msg))
    print("\nlive final answer:", live_result["messages"][-1].content)
else:
    print("No ANTHROPIC_API_KEY set — skipping the live run.")
    print("Set it in your environment or repo-root .env to drive Sonnet 4.6.")

No ANTHROPIC_API_KEY set — skipping the live run.
Set it in your environment or repo-root .env to drive Sonnet 4.6.


[↑ Back to top](#top)

## 9. Takeaways

- **`create_agent` is the L10 loop, packaged.** One call — model + tools + system prompt — returns a running shallow agent. No `while`, no branch, no message bookkeeping written by you.
- **Behavioral equivalence, proven by the run.** Same tools, same order (`calculator` then `lookup`), same natural stop as L10 — from a single constructor call.
- **The returned `messages` is the run.** Same user / assistant-with-tool-calls / tool-result / assistant sequence L10 built by hand; the final answer is the last message. This message history is exactly what [L12](../L12/objectives.md) reads as a *trace* and [L13](../L13/objectives.md) judges with an *eval set*.
- **The graph it renders matches the diagram you drew** — `agent`, `tools`, the back-edge, the conditional exit. The one-liner is not magic; it is the picture.
- **The step cap didn't go away** — it's the framework's recursion limit now (default 25). Hitting it is still a signal worth investigating.
- **Same code, live or offline.** Swap `FakeModel` for `ChatAnthropic` and nothing else changes — the interchangeability you learned in L10.

[↑ Back to top](#top)